# 04 · 참조 핸들 — 버리지 않고 치워두기

앞의 압축들은 **무엇이 중요한지 미리 맞혀야** 했습니다.

- 02번 형식 압축은 안전했지만 줄이는 데 한계가 있었습니다
- 03번 요약은 많이 줄였지만 **버린 것은 영영 못 찾았습니다**

무엇이 필요한지는 **질문이 오기 전에는 알 수 없습니다.** 그래서 딜레마가 생깁니다.

| | 위험 | 절감 |
|---|---|---|
| 세게 압축 | 필요한 걸 버렸을 수 있습니다 | 큼 |
| 약하게 압축 | 안전합니다 | 조금 |

### 발상을 바꿉니다 — 버리지 말고 치워두기

```
압축    긴 블록을 짧은 자리표시자로 바꿉니다   →  [60개 중 5개 표시 · hash=abc123]
보관    원문은 별도 저장소에 그대로 둡니다      →  버리지 않습니다
되찾기  모델이 필요하면 도구로 꺼내옵니다       →  fetch_original(hash="abc123")
```

**틀려도 복구할 수 있으니 세게 압축해도 됩니다.**
다만 되찾으려면 **API 호출이 한 번 더** 필요합니다. 그래서 핵심 질문은 이것입니다.

> **몇 퍼센트나 되찾으면 손해로 뒤집힐까요?**

### 전부 직접 만듭니다

외부 압축 라이브러리를 쓰지 않습니다. 부품은 세 개뿐입니다.

- **저장소** — 해시 키로 원문을 넣고 꺼내는 딕셔너리
- **압축 함수** — 블록을 줄이고 자리표시자를 남기는 함수
- **도구 루프** — 모델이 도구를 부르면 저장소에서 꺼내 돌려주는 반복문

> 이 패턴을 Headroom 은 **CCR(Compress-Cache-Retrieve)** 이라고 부릅니다.
> 용어 대응표는 맨 끝 정리 절에 있습니다.

## 0-1. 전체 흐름 한눈에

아래 숫자는 이 노트북에서 **실제로 측정한 값**입니다.

### 공통 준비 — 여기까지는 항상 같습니다

| 단계 | 하는 일 | 컨텍스트 크기 |
|---|---|---|
| ① 도구 출력 | 주문 60건 JSON 이 들어옵니다 | 7,377자 · 약 2,900토큰 |
| ② 압축·보관 | 원문은 **저장소로**, 컨텍스트에는 **자리표시자만** | **약 241토큰 (92% 감소)** |
| ③ 모델 호출 | 자리표시자 + 되찾기 도구를 함께 보냅니다 | — |

②에서 원문을 **버리는 게 아니라 옮겨둡니다.** 그래서 무손실입니다.

### ③ 이후 — 모델의 판단에 따라 두 갈래로 나뉩니다

**A. 자리표시자만으로 답할 수 있는 경우** · API 1회

| 호출 | 보내는 것 | 모델이 하는 일 | 입력 토큰 |
|---|---|---|---|
| 1차 | 자리표시자 + 질문 | 바로 답변 | **358** |

**B. 원문이 필요한 경우** · API 2회 (= 왕복 1회 추가)

| 호출 | 보내는 것 | 모델이 하는 일 | 입력 토큰 |
|---|---|---|---|
| 1차 | 자리표시자 + 질문 | 답 대신 `fetch_original("7f3a")` 호출 | 319 |
| — | *우리가 그 호출을 가로채 저장소에서 원문을 꺼냅니다* | | — |
| 2차 | **원문 전체** + 질문 | 답변 | 2,657 |
| | | **합계** | **2,976** |

이 노트북에서 **'왕복'** 은 이 B 처럼 **API 를 두 번 부르는 것**을 뜻합니다.

### 그래서 이득 조건은 하나입니다

같은 조건에서 원문을 그냥 다 넣으면 **2,982토큰**입니다 (3절 비교표 Q1 행).

| 경로 | 입력 토큰 | 전체 투입 대비 |
|---|---|---|
| **A** 자리표시자로 해결 | 358 | **−88%** |
| **B** 원문을 되찾음 | 2,976 | 거의 본전 + 왕복 지연 |

> **A 로 가는 비율이 높을수록 이득입니다.**
> 에이전트의 도구 출력(파일 목록·로그·검색 결과)이 여기에 맞습니다.
> 양은 많은데 실제로 열어보는 건 일부뿐이기 때문입니다.

## 준비 1 · 설정

01~03번과 같습니다. 여기에 **도구 호출(function calling)** 이 추가됩니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 실험 환경을 갖춥니다.
#   · responses()  도구(tools) 파라미터를 넘길 수 있게 합니다
#   · call_tokens() 한 번의 호출에서 쓴 input/output 토큰을 돌려줍니다
# API 호출: 0회 (준비만)
# ──────────────────────────────────────────────────────────────────────────
import hashlib, json, os, re, shutil, subprocess, time
import urllib.request, urllib.error
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

from nbtools import Usage, Price, show_table

load_dotenv(find_dotenv(usecwd=True) or str(Path.cwd() / ".env"), override=False)


def require(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(f"{name} 가 없습니다. `cp .env.example .env` 후 값을 채워주세요.")
    return v


ENDPOINT   = require("AZURE_OPENAI_ENDPOINT").rstrip("/")
DEPLOYMENT = require("AZURE_OPENAI_DEPLOYMENT")

AZ_CANDIDATES = [os.environ.get("AZ_CLI"), shutil.which("az"),
                 "/opt/homebrew/bin/az", "/usr/local/bin/az",
                 str(Path.home() / ".local/bin/az")]


def find_az():
    for c in AZ_CANDIDATES:
        if c and Path(c).exists():
            return c
    raise RuntimeError("az CLI 를 찾지 못했습니다. .env 에 AZ_CLI=/전체/경로/az 를 넣어주세요.")


def auth_headers():
    key = os.environ.get("AZURE_OPENAI_API_KEY")
    if key:
        return {"api-key": key}
    r = subprocess.run([find_az(), "account", "get-access-token",
                        "--scope", "https://cognitiveservices.azure.com/.default", "-o", "json"],
                       capture_output=True, text=True, timeout=90)
    if r.returncode != 0:
        raise RuntimeError(f"az 토큰 발급에 실패했습니다.\n{r.stderr.strip()[:300]}")
    return {"Authorization": "Bearer " + json.loads(r.stdout)["accessToken"]}


HEADERS = auth_headers()


def responses(payload):
    req = urllib.request.Request(
        f"{ENDPOINT}/openai/v1/responses?api-version=preview",
        data=json.dumps({"model": DEPLOYMENT, **payload}).encode(),
        headers={"Content-Type": "application/json", **HEADERS}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8','replace')[:400]}")


def rtext(resp):
    out = []
    for item in resp.get("output", []):
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text"):
                out.append(c.get("text", ""))
    return "".join(out).strip()


def tool_calls(resp):
    return [it for it in resp.get("output", []) if it.get("type") == "function_call"]


print("배포:", DEPLOYMENT, "· 준비 완료")

## 준비 2 · 실험 대상 — 에이전트 도구 출력

이 기법이 겨냥하는 것은 산문이 아니라 **도구가 뱉어내는 대량 출력**입니다.

- 파일 목록 500개
- 로그 수천 줄
- API 응답 JSON 배열

이런 출력은 **대부분 안 쓰이는데 컨텍스트를 다 잡아먹습니다.**
질문 하나에 실제로 필요한 건 그중 몇 줄뿐인 경우가 많습니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 에이전트 도구 출력 두 가지를 만듭니다.
#   · ORDERS  주문 60건 JSON 배열 — 대량 구조 데이터
#   · LOGS    파드 로그 80줄 — 대량 텍스트
#   · 실제 에이전트가 파일 읽기·검색으로 받아오는 것과 같은 모양입니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
ORDERS = [
    {"order_id": f"A-{1000+i}",
     "status": ["paid", "shipping", "refunded", "cancelled"][i % 4],
     "amount": 10000 + (i * 1373) % 90000,
     "created_at": f"2026-03-{(i % 28) + 1:02d}",
     "channel": ["app", "web", "store"][i % 3]}
    for i in range(60)
]
ORDERS_JSON = json.dumps(ORDERS, ensure_ascii=False, indent=1)

LOG_LINES = []
for i in range(80):
    ts = f"2026-03-14 0{i//30}:{i%60:02d}:11"
    if i == 47:
        LOG_LINES.append(f"{ts} ERROR nodepool-gpu-a100 OOMKilled container=infer memory=39.8Gi limit=40Gi")
    elif i % 17 == 0:
        LOG_LINES.append(f"{ts} WARN  readiness probe failed attempt={i//17} endpoint=/healthz")
    else:
        LOG_LINES.append(f"{ts} INFO  request served path=/v1/chat status=200 latency_ms={40 + i % 30}")
LOGS = "\n".join(LOG_LINES)

show_table(
    ["도구 출력", "항목", "문자수"],
    [["주문 목록 JSON", f"{len(ORDERS)}건", f"{len(ORDERS_JSON):,}자"],
     ["파드 로그",      f"{len(LOG_LINES)}줄", f"{len(LOGS):,}자"]],
    align=["left", "right", "right"],
    title="실험 대상",
    note="질문 하나에 실제로 필요한 것은 이 중 한두 줄뿐인 경우가 많습니다.",
)

## 1. 압축과 보관 — 자리표시자로 바꾸고 원문은 저장소에

두 가지를 동시에 합니다.

1. 블록을 짧게 줄입니다 (JSON 배열은 앞 몇 개만, 로그는 앞뒤 몇 줄만)
2. 원문을 **해시 키로 저장소에 보관**합니다
3. 줄인 자리에 **자리표시자**를 남깁니다

자리표시자에는 세 가지가 들어가야 합니다.

```
[60개 중 5개만 표시. 전체를 보려면 retrieve: hash=a1b2c3d4]
  └ 얼마나 줄었는지    └ 어떻게 되찾는지        └ 어떤 키인지
```

이 정보가 있어야 모델이 **"더 볼 게 있구나"** 를 알고 스스로 판단할 수 있습니다.
자리표시자 없이 그냥 잘라내면 모델은 잘린 줄도 모릅니다.

**압축 결과 = 남긴 데이터 + 자리표시자 한 줄** 입니다.

```
[{"order_id":"A-1000", ...}, ... 앞 5건 ...]            ← 남긴 데이터 (540자)
[60개 중 5개만 표시. 전체를 보려면 retrieve: hash=011d6d2a]  ← 자리표시자 (47자)
```

해시는 자리표시자 안의 **8글자**일 뿐입니다.
절감의 실체는 **"60건을 5건으로 줄인 것"** 이고, 해시는 나머지 55건을 되찾을 열쇠입니다.

> **Headroom 참고** — 같은 아이디어를 제품화한 Headroom 은 이 자리표시자를 *marker* 라 부르고
> `[1000 items compressed to 20. Retrieve more: hash=abc123]` 형식을 씁니다. 구성은 동일합니다.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 압축·보관 단계를 직접 구현합니다.
#   · compress_json  배열 앞 N개만 남기고 나머지는 마커로
#   · compress_log   앞뒤 몇 줄만 남기고 가운데는 마커로
#   · 원문은 store 에 해시 키로 보관합니다 (버리지 않습니다)
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
class BlockStore:
    """압축한 원문을 해시 키로 보관합니다. 실제 시스템은 보통 LRU 캐시를 씁니다."""

    def __init__(self):
        self.cache = {}
        self.hits = 0

    def _put(self, text):
        h = hashlib.sha256(text.encode()).hexdigest()[:8]
        self.cache[h] = text
        return h

    def compress_json(self, rows, keep=5):
        """JSON 배열 → 앞 keep개 + 마커. 원문 전체는 캐시에 보관합니다."""
        full = json.dumps(rows, ensure_ascii=False, indent=1)
        h = self._put(full)
        head = json.dumps(rows[:keep], ensure_ascii=False)
        return (f"{head}\n"
                f"[{len(rows)}개 중 {keep}개만 표시. 전체를 보려면 retrieve: hash={h}]"), h

    def compress_log(self, text, head_n=3, tail_n=3):
        """로그 → 앞뒤 몇 줄 + 마커. 가운데가 잘려도 원문은 남아 있습니다."""
        lines = text.splitlines()
        h = self._put(text)
        kept = lines[:head_n] + [f"[... {len(lines)-head_n-tail_n}줄 생략. "
                                 f"전체를 보려면 retrieve: hash={h}]"] + lines[-tail_n:]
        return "\n".join(kept), h

    def retrieve(self, h):
        self.hits += 1
        return self.cache.get(h, f"(hash={h} 를 찾을 수 없습니다)")


store = BlockStore()
ORDERS_C, H_ORDERS = store.compress_json(ORDERS, keep=5)
LOGS_C,   H_LOGS   = store.compress_log(LOGS)


def split_marker(compressed):
    """압축 결과를 '남긴 데이터' 와 '자리표시자' 로 나눕니다."""
    lines = compressed.splitlines()
    mark = next(l for l in lines if "hash=" in l)
    kept = "\n".join(l for l in lines if l is not mark)
    return kept, mark


kept_o, mark_o = split_marker(ORDERS_C)
kept_l, mark_l = split_marker(LOGS_C)

show_table(
    ["블록", "원본", "남긴 데이터", "자리표시자", "압축 후 합계", "절감"],
    [["주문 JSON", f"{len(ORDERS_JSON):,}자", f"{len(kept_o):,}자",
      f"{len(mark_o)}자", f"{len(ORDERS_C):,}자", f"{1-len(ORDERS_C)/len(ORDERS_JSON):.0%}"],
     ["파드 로그", f"{len(LOGS):,}자", f"{len(kept_l):,}자",
      f"{len(mark_l)}자", f"{len(LOGS_C):,}자", f"{1-len(LOGS_C)/len(LOGS):.0%}"]],
    align=["left", "right", "right", "right", "right", "right"],
    title="압축 결과의 구성",
    note="압축 후 = 남긴 데이터 + 자리표시자 한 줄. "
         "해시는 자리표시자 안의 8글자일 뿐이고, 절감의 실체는 '60건 -> 5건' 입니다.",
)

def show_before_after(name, original, compressed, h, head_chars=220):
    """압축 전후를 나란히 보여줍니다."""
    kept, mark = split_marker(compressed)
    print("\n" + "=" * 78)
    print(f"  {name}")
    print("=" * 78)

    print(f"\n[압축 전]  {len(original):,}자")
    print("-" * 78)
    print(original[:head_chars].rstrip() + "  …(이하 생략)")

    print(f"\n[압축 후]  {len(compressed):,}자   ({1-len(compressed)/len(original):.0%} 감소)")
    print("-" * 78)
    print(f"  ▸ 남긴 데이터 {len(kept):,}자")
    for line in kept[:head_chars].rstrip().splitlines():
        print(f"    {line}")
    if len(kept) > head_chars:
        print("    …(이하 생략)")
    print(f"\n  ▸ 자리표시자 {len(mark)}자")
    print(f"    {mark}")

    print(f"\n[저장소]  원문 {len(original):,}자가 그대로 보관돼 있습니다")
    print("-" * 78)
    print(f'    store.cache["{h}"]  →  {len(store.cache[h]):,}자')


show_before_after("① 주문 목록 JSON  ·  앞 5건만 남기고 나머지는 자리표시자로",
                  ORDERS_JSON, ORDERS_C, H_ORDERS)

show_before_after("② 파드 로그  ·  앞뒤 3줄만 남기고 가운데는 자리표시자로",
                  LOGS, LOGS_C, H_LOGS)

print("\n" + "=" * 78)
print("  두 경우 모두 원문은 하나도 사라지지 않았습니다. 컨텍스트에서만 빠졌습니다.")
print("=" * 78)

## 2. 되찾기 — 도구를 쥐여주고 왕복을 관찰합니다

모델에게 되찾기 도구를 정의해서 넘깁니다.

```json
{
  "type": "function",
  "name": "fetch_original",
  "description": "자리표시자에 있는 hash 로 원본 전체 데이터를 되찾습니다.",
  "parameters": { "hash": "자리표시자의 hash 값" }
}
```

그다음 **0-1 절의 A/B 갈림길**을 코드로 구현합니다.
모델이 도구를 부르면 우리가 가로채 저장소에서 꺼내주고, 2차 호출을 이어갑니다.

**되찾으라고 강제하지 않습니다.** 스스로 판단하게 두고 실제로 어떻게 하는지 봅니다.

> **Headroom 참고** — Headroom 은 이 도구를 `headroom_retrieve(hash)` 로 **자동 주입**하고
> 도구 호출도 프록시가 가로채 처리합니다. 아래 루프가 그 프록시가 하는 일입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 참조 핸들 실행 루프를 직접 구현합니다. 모델이 도구를 부르면 자동으로 원문을 넣어줍니다.
#   · 1차 호출 → 도구 호출이 오면 → 캐시에서 꺼내 → 2차 호출
#   · 모든 호출의 input/output 토큰을 합산합니다 (왕복 비용을 숨기지 않습니다)
#   · retrieved 에 어떤 해시를 되찾았는지 기록합니다
# API 호출: 0회 (정의만)
# ──────────────────────────────────────────────────────────────────────────
RETRIEVE_TOOL = [{
    "type": "function",
    "name": "fetch_original",
    "description": "압축 마커에 있는 hash 로 원본 전체 데이터를 되찾습니다.",
    "parameters": {
        "type": "object",
        "properties": {"hash": {"type": "string", "description": "마커의 hash 값"}},
        "required": ["hash"],
        "additionalProperties": False,
    },
}]

SYSTEM = ("당신은 운영 데이터를 분석하는 어시스턴트입니다. "
          "제공된 내용만으로 답할 수 없으면 fetch_original 로 원본을 되찾아 정확히 답하세요. "
          "추측하지 마세요.")


def run_handle(question, blocks, max_rounds=3):
    """참조 핸들 루프.

    돌려주는 것: (답변, 총입력토큰, 총출력토큰, 되찾은 해시들, 호출별 내역)

    ★ 여기서 '왕복' 이란 API 를 두 번 부르는 것을 말합니다.
        1차: 자리표시자만 보냄        -> 모델이 "원문 주세요" 라며 도구를 부름 (답을 안 줌)
        2차: 원문을 붙여서 다시 보냄  -> 그제서야 답변
      2차 호출에는 원문이 통째로 들어가므로 입력 토큰이 크게 뜁니다.
    """
    convo = f"{SYSTEM}\n\n{blocks}\n\n질문: {question}"
    payload = {"input": convo, "tools": RETRIEVE_TOOL, "max_output_tokens": 400}
    tin = tout = 0
    retrieved, trace = [], []

    for _ in range(max_rounds):
        r = responses(payload)
        u = Usage.from_response(r, model=DEPLOYMENT)
        tin += u.input_tokens; tout += u.output_tokens

        tc = tool_calls(r)
        trace.append({"input": u.input_tokens, "output": u.output_tokens,
                      "did": "도구 호출" if tc else "답변"})

        if not tc:
            return rtext(r), tin, tout, retrieved, trace

        # 도구 호출을 가로채 캐시에서 꺼내 돌려줍니다 (모델은 이 과정을 모릅니다)
        outputs = []
        for c in tc:
            h = json.loads(c["arguments"]).get("hash", "")
            retrieved.append(h)
            outputs.append({"type": "function_call_output",
                            "call_id": c["call_id"], "output": store.retrieve(h)})
        payload = {"previous_response_id": r["id"], "input": outputs,
                   "tools": RETRIEVE_TOOL, "max_output_tokens": 400}

    return "(왕복 한도 초과)", tin, tout, retrieved, trace


def run_full(question, blocks):
    """비교용 — 원문을 통째로 넣습니다. 도구가 없으니 항상 1회로 끝납니다."""
    r = responses({"input": f"{SYSTEM}\n\n{blocks}\n\n질문: {question}",
                   "max_output_tokens": 400})
    u = Usage.from_response(r, model=DEPLOYMENT)
    return rtext(r), u.input_tokens, u.output_tokens, [], [{"input": u.input_tokens}]


print("참조 핸들 루프 준비 완료 · 도구:", RETRIEVE_TOOL[0]["name"])

### 질문 세 가지로 왕복을 관찰합니다

되찾기가 필요한 정도가 다른 질문을 골랐습니다.

| 질문 | 예상 |
|---|---|
| Q1 첫 주문의 상태 | 자리표시자 위쪽 5건에 이미 있습니다 → **왕복 없음** |
| Q2 OOMKilled 발생 시각 | 로그 가운데가 잘렸습니다 → **왕복 발생** |
| Q3 환불 건수 합계 | 60건 전체를 세야 합니다 → **왕복 발생** |

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 세 질문을 실행하고, 왕복이 실제로 어떻게 일어나는지 호출 단위로 봅니다.
#   · 모델이 스스로 판단해서 도구를 부릅니다 — 강제하지 않습니다
#   · 1차에서 답하면 왕복 없음, 도구를 부르면 2차 호출이 이어집니다
# API 호출: 질문 3개 × (1~2회) ≈ 5회 · 약 20초
# ──────────────────────────────────────────────────────────────────────────
QUESTIONS = [
    ("Q1 첫 주문", "표시된 주문 중 첫 번째 주문의 주문번호와 상태는 무엇인가요?", ORDERS_C),
    ("Q2 장애 시각", "로그에서 OOMKilled 가 발생한 시각과 컨테이너 이름은 무엇인가요?", LOGS_C),
    ("Q3 환불 집계", "전체 주문 중 status 가 refunded 인 건수는 총 몇 건인가요?", ORDERS_C),
]

results = []
for label, q, blk in QUESTIONS:
    ans, tin, tout, got, trace = run_handle(q, blk)
    results.append(dict(label=label, q=q, blk=blk, ans=ans,
                        tin=tin, tout=tout, got=got, trace=trace))
    time.sleep(0.3)

# 호출 하나가 한 줄입니다. 두 줄짜리 질문이 '왕복' 이 일어난 것입니다.
rows = []
for r in results:
    for n, t in enumerate(r["trace"], start=1):
        rows.append([r["label"] if n == 1 else "",
                     f"{n}차", f"{t['input']:,}", t["did"]])
    rows.append(["", "합계", f"{r['tin']:,}", ""])

show_table(
    ["질문", "호출", "입력 토큰", "모델이 한 일"],
    rows,
    align=["left", "left", "right", "left"],
    title="호출 단위로 본 실행 과정",
    note="1차에서 '도구 호출' 이 나오면 아직 답을 못 준 것입니다. "
         "그래서 원문을 붙여 2차 호출을 한 번 더 합니다 — 이것이 '왕복' 입니다.",
)

# 질문 하나를 골라 흐름을 문장으로 따라가 봅니다
for r in results:
    print(f"\n【{r['label']}】 {r['q']}")
    for n, t in enumerate(r["trace"], start=1):
        if t["did"] == "도구 호출":
            print(f"   {n}차 호출  자리표시자 + 질문을 보냄        → {t['input']:>6,}토큰 과금")
            print(f"            모델: \"답 대신 원문을 주세요\"      (fetch_original 호출)")
            print(f"            우리: 저장소에서 원문을 꺼내 붙임")
        else:
            what = "원문 전체 + 질문" if n > 1 else "자리표시자 + 질문"
            print(f"   {n}차 호출  {what:<22s} → {t['input']:>6,}토큰 과금")
            print(f"            모델: 답변 → {r['ans'][:32]}…")
    print(f"   {'─'*54}")
    print(f"   누적 과금 입력 {r['tin']:,}토큰"
          f"   ({'왕복 없음' if len(r['trace'])==1 else '왕복 1회'})")

## 3. 손익분기 — 몇 퍼센트를 되찾으면 손해가 될까요

전체를 넣는 방식과 나란히 재봅니다.

> **숫자 출처 주의** — 이 절에는 비슷하지만 다른 값이 두 개 나옵니다.
> **2,918** 은 JSON 원문 *텍스트만* 잰 값이고,
> **2,982** 는 거기에 *시스템 지침과 질문까지* 더한 값입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 같은 질문을 '전체 투입' 방식으로도 실행해 직접 비교합니다.
#   · 핸들 방식은 왕복 비용까지 포함한 누적 입력 토큰으로 비교합니다
#   · 되찾은 질문과 안 되찾은 질문이 어떻게 갈리는지 봅니다
# API 호출: 질문 3개 = 3회 · 약 15초
# ──────────────────────────────────────────────────────────────────────────
rows = []
sum_ccr = sum_full = 0

for r in results:
    full_blk = ORDERS_JSON if r["blk"] is ORDERS_C else LOGS
    _, ftin, _, _, _ = run_full(r["q"], full_blk)
    sum_ccr += r["tin"]; sum_full += ftin
    delta = (r["tin"] - ftin) / ftin
    rows.append([r["label"], "O" if r["got"] else "—",
                 f"{ftin:,}", f"{r['tin']:,}", f"{delta:+.0%}",
                 "이득" if delta < 0 else "손해"])
    time.sleep(0.3)

show_table(
    ["질문", "되찾기", "전체 투입", "핸들 방식", "차이", "판정"],
    rows,
    align=["left", "center", "right", "right", "right", "left"],
    foot=["합계", "", f"{sum_full:,}", f"{sum_ccr:,}",
          f"{(sum_ccr-sum_full)/sum_full:+.0%}", ""],
    title="참조 핸들 vs 전체 투입 (누적 입력 토큰)",
    note="되찾지 않은 질문은 크게 이득이고, 되찾은 질문은 왕복 때문에 손해로 뒤집힙니다.",
)

### 손익분기식

되찾기 비율을 **r** 이라 하면 대략 이렇게 됩니다.

```
핸들 비용  ≈  (1−r) × 자리표시자토큰  +  r × (자리표시자 + 원문 + 왕복오버헤드)
전체 비용  ≈  원문토큰
```

정리하면 **원문이 클수록, 되찾는 비율이 낮을수록 유리**합니다.

| 상황 | 판단 |
|---|---|
| 큰 도구 출력을 붙이는데 대부분 안 열어봄 | **크게 유리합니다** |
| 매번 전부 열어봄 | **손해입니다** — 왕복만 늘어납니다 |
| 원문이 애초에 작음 | 자리표시자 비용이 아까워 의미가 없습니다 |
| 지연에 민감한 실시간 응답 | 왕복 1회가 부담이라 신중해야 합니다 |

아래에서 되찾기 비율을 바꿔가며 손익이 뒤집히는 지점을 계산해 봅니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 되찾기 비율을 0~100% 로 바꿔가며 손익분기점을 계산합니다.
#   · 실제 측정한 마커·원문·왕복 토큰을 그대로 씁니다 (가정값 아님)
#   · 부호가 바뀌는 지점이 손익분기입니다
# API 호출: 3회 (마커/원문 토큰 실측)
# ──────────────────────────────────────────────────────────────────────────
def toks(text):
    r = responses({"input": text, "max_output_tokens": 16})
    return Usage.from_response(r, model=DEPLOYMENT).input_tokens


t_marker = toks(ORDERS_C)      # 마커만 보낼 때
t_full   = toks(ORDERS_JSON)   # 원문 전체를 보낼 때
# 되찾을 때는 1차(마커) + 2차(원문 + 도구 왕복) 이 모두 과금됩니다
r3 = next(x for x in results if x["got"])
t_round = r3["tin"]

print("※ 여기서는 '텍스트 자체' 의 토큰만 잽니다.")
print("   시스템 지침·질문이 붙으면 60토큰 남짓 더 늘어납니다(앞 비교표의 값이 조금 큰 이유).\n")
print(f"자리표시자만 보낼 때    {t_marker:,} 토큰")
print(f"원문 전체를 보낼 때     {t_full:,} 토큰")
print(f"되찾기 왕복까지 갈 때   {t_round:,} 토큰  (1차 + 2차 누적, 지침·질문 포함)\n")

rows = []
for pct in (0, 10, 25, 50, 75, 100):
    r = pct / 100
    handle_cost = (1 - r) * t_marker + r * t_round
    diff = (handle_cost - t_full) / t_full
    rows.append([f"{pct}%", f"{handle_cost:,.0f}", f"{t_full:,}", f"{diff:+.0%}",
                 "이득" if diff < 0 else "손해"])

show_table(
    ["되찾기 비율", "핸들 평균 토큰", "전체 투입", "차이", "판정"],
    rows,
    align=["right", "right", "right", "right", "left"],
    title="되찾기 비율에 따른 손익",
    note="부호가 바뀌는 지점이 손익분기입니다. 이 값은 원문 크기에 따라 달라집니다.",
)

## 4. 무손실 증명

02번에서 했던 것과 같은 검증입니다. **되찾은 것이 원문과 완전히 같아야** 무손실입니다.

03번의 요약과 결정적으로 다른 지점입니다.

- **03번 요약** — 원문을 버렸으므로 요약에 빠진 정보는 영영 못 찾습니다
- **04번 참조 핸들** — 원문이 저장소에 있으므로 **언제든 그대로 복구**됩니다

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 되찾은 내용이 원문과 바이트 단위로 같은지 확인합니다.
#   · 압축률이 높아도 원문은 손상되지 않아야 합니다
#   · 이것이 '가역(reversible)' 의 정의입니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
checks = [
    ("주문 JSON", ORDERS_JSON, store.retrieve(H_ORDERS)),
    ("파드 로그", LOGS,        store.retrieve(H_LOGS)),
]

show_table(
    ["블록", "원문", "되찾은 것", "동일한가"],
    [[n, f"{len(a):,}자", f"{len(b):,}자", "예" if a == b else "아니오"]
     for n, a, b in checks],
    align=["left", "right", "right", "center"],
    title="가역성 검증",
    note="'예' 여야 무손실입니다. 압축률이 아무리 높아도 원문은 그대로 남아 있습니다.",
)

print(f"캐시에 보관된 블록 {len(store.cache)}개 · 되찾기 호출 {store.hits}회")

## 5. 캐시와의 관계 — 03번의 문제를 피하는 방법

03번에서 이런 문제를 만났습니다.

> 요약으로 대체하면 대화 앞부분이 바뀌고, **프롬프트 캐시가 전부 무효**가 됩니다.

참조 핸들은 **압축 대상을 뒤쪽으로 제한해서** 이걸 피할 수 있습니다.

```
[시스템 프롬프트][도구 정의][이전 턴들]   [최신 도구 출력]
└──────── 그대로 둡니다 (캐시 유지) ───┘  └─ 여기만 압축 ─┘
```

02번 결론과 같습니다 — **고정부는 건드리지 말고 가변부만 압축한다.**

> **Headroom 참고** — Headroom 은 앞부분을 *hot zone*, 최신 블록을 *live zone* 이라 부르고
> **live zone 만 압축**합니다. 같은 이유입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: hot zone 을 보존하면 압축을 해도 캐시가 유지되는지 실측합니다.
#   · 앞에 1,024토큰 넘는 고정 시스템 지침을 둡니다 (캐시 발동 조건)
#   · 뒤쪽 도구 출력만 압축본으로 바꿔가며 3회 호출합니다
#   · ★ RUN_ID 로 매번 찬 캐시에서 시작합니다 (01번 8절과 같은 이유)
# API 호출: 6회 · 약 30초
# ──────────────────────────────────────────────────────────────────────────
import uuid
RUN_ID = uuid.uuid4().hex[:8]

HOT = (f"[run={RUN_ID}] 당신은 운영 데이터를 분석하는 어시스턴트입니다.\n"
       + "".join(f"규칙{i}. 답변은 주어진 데이터에 있는 값만 사용하고 추측하지 않으며, "
                 f"수치와 단위를 원문 그대로 표기합니다.\n" for i in range(1, 61)))


def cache_run(label, live_block):
    rows = []
    for t in range(1, 4):
        r = responses({"input": f"{HOT}\n\n[도구 출력]\n{live_block}\n\n질문: 첫 주문 상태는?",
                       "max_output_tokens": 16})
        u = Usage.from_response(r, model=DEPLOYMENT)
        rows.append([f"{t}회차", f"{u.input_tokens:,}", f"{u.cached_tokens:,}",
                     f"{u.billed_input:,}", f"{u.cache_hit_rate:.1%}"])
        time.sleep(1.5)
    show_table(["회차", "입력", "캐시적중", "과금입력", "적중률"], rows,
               align=["left", "right", "right", "right", "right"], title=label)
    return rows


cache_run("앞부분 보존 + 뒷부분만 압축", ORDERS_C)
cache_run("hot zone 보존 + 원문 전체 (압축 안 함)", ORDERS_JSON)

print("두 경우 모두 앞쪽 hot zone 이 동일하므로 캐시가 맞습니다.")
print("=> 압축을 하면서도 캐시를 유지할 수 있습니다. 압축 위치를 뒤로 몰았기 때문입니다.")

## 6. 정리

### 다른 방법들과의 자리

| | 원문을 | 되돌리기 | 압축 비용 | 추가 대가 |
|---|---|---|---|---|
| 02 형식 압축 | 형식만 바꿉니다 | 가능 | 0 | 없습니다 |
| 03 대화 요약 | **버립니다** | 불가 | LLM 호출 | 정보 손실 |
| **04 참조 핸들** | **치워둡니다** | **가능** | 0 | **왕복 1회 · 지연** |

### 배운 것

- **"무엇이 중요한지 미리 맞히는" 문제가 사라집니다** — 틀려도 되찾을 수 있습니다
- **공짜가 아닙니다** — 되찾으면 왕복이 추가되고 원문이 결국 컨텍스트에 들어옵니다
- **되찾기 비율이 손익을 가릅니다** — 대부분 안 열어볼수록 유리합니다
- **원문이 클수록 유리합니다** — 자리표시자 비용이 상대적으로 작아집니다
- **압축 위치를 뒤로 몰면 캐시를 지킬 수 있습니다** — 앞부분은 건드리지 않습니다

### Headroom 용어 대응표

나중에 `labs/07-headroom/` 에서 실제 라이브러리를 붙일 때 참고용입니다.

| 이 노트북 | Headroom | 비고 |
|---|---|---|
| `BlockStore` | Compression Store | Headroom 은 LRU 캐시를 씁니다 |
| 자리표시자 | marker | `hash=...` 를 담는 형식은 동일합니다 |
| `fetch_original` | `headroom_retrieve` | 도구를 자동 주입해 줍니다 |
| 도구 호출 가로채기 | Response Handler | 프록시가 처리해 앱은 모릅니다 |
| 앞부분 / 최신 블록 | hot zone / live zone | live zone 만 압축합니다 |
| `compress_json` | SmartCrusher | JSON 배열 전용 압축기 |
| `compress_log` | ContentRouter | 코드·로그·텍스트를 종류별로 처리 |
| (미구현) | Context Tracker | 모델이 묻기 전에 미리 펼칩니다 |

### 남은 숙제

- **선제적 펼치기** — 무엇을 압축했는지 기억해 두고, 관련 질문이 오면
  **모델이 묻기 전에 미리 펼치는** 방법. 왕복 지연을 줄입니다
- **되찾기 비율 측정** — 실제 트래픽에서 r 이 얼마인지 재봐야 도입 여부를 판단할 수 있습니다